## In this file 

In [ ]:
import pandas as pd
import numpy as np


In [13]:
## Path Configuration


DEV_IN_PATH  = "../../data/raw/development.csv"
EVAL_IN_PATH = "../../data/raw/evaluation.csv"

DEV_OUT_PATH  = "../../data/processed/development_processed.csv"
EVAL_OUT_PATH = "../../data/processed/evaluation_processed.csv"


OUT_DIR.mkdir(parents=True, exist_ok=True)


In [14]:
## Load data from CSV file
df_dev  = pd.read_csv(DEV_IN_PATH)
df_eval = pd.read_csv(EVAL_IN_PATH)

for df in (df_dev, df_eval):
	df["article"] = df["article"].fillna("").astype(str)
	df["title"]   = df["title"].fillna("").astype(str)
	df["source"]  = df["source"].fillna("").astype(str)


In [15]:
## Text FEATURE ENGINEERING

def build_text(df):
	"""
	Canonical textual representation shared by all models.
	"""
	return (df["title"] + " " + df["article"]).str.lower()

df_dev["text"]  = build_text(df_dev)
df_eval["text"] = build_text(df_eval)

In [16]:
## Numerical FEATURE ENGINEERING

def add_numeric_features(df):
	df = df.copy()

	df["n_tokens"]    = df["article"].str.split().str.len()
	df["title_len"]   = df["title"].str.len()
	df["article_len"] = df["article"].str.len()
	df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)

	num_cols = ["n_tokens", "title_len", "article_len", "title_ratio"]
	df[num_cols] = df[num_cols].replace([np.inf, -np.inf], 0).fillna(0)

	return df

df_dev  = add_numeric_features(df_dev)
df_eval = add_numeric_features(df_eval)

In [17]:
# Columns Selection

BASE_COLS = [
	"Id",
	"source",
	"text",
	"n_tokens",
	"title_len",
	"article_len",
	"title_ratio"
]

DEV_COLS = BASE_COLS + ["label"]

df_dev_out  = df_dev[DEV_COLS]
df_eval_out = df_eval[BASE_COLS]

In [18]:
# Saving File 
df_dev_out.to_csv(DEV_OUT_PATH, index=False)
df_eval_out.to_csv(EVAL_OUT_PATH, index=False)

print("Preprocessing completed.")
print("Saved:")
print(f" - {DEV_OUT_PATH}  | shape = {df_dev_out.shape}")
print(f" - {EVAL_OUT_PATH} | shape = {df_eval_out.shape}")

Preprocessing completed.
Saved:
 - ../../data/processed/development_processed.csv  | shape = (79997, 8)
 - ../../data/processed/evaluation_processed.csv | shape = (20000, 7)


In [20]:
df_dev_out.head()


,Id,source,text,n_tokens,title_len,article_len,title_ratio,label
0,0,AllAfrica.com,opec boosts nigeria&#39;s oil revenue by .82m ...,35,49,214,0.227907,5
1,1,Xinhua,yearender: mideast peace roadmap reaches dead-...,27,57,161,0.351852,0
2,2,Yahoo,battleground dispatches for oct. 5 \\n (cqp...,30,59,181,0.324176,0
3,3,BBC,air best to resuscitate newborns air rather th...,18,32,111,0.285714,0
4,4,Yahoo,high tech german train crash kills at least on...,79,65,720,0.090153,0
